In [1]:
"Verify the dataset before training the models" 

'Verify the dataset before training the models'

In [9]:
import torch
from torch.utils.data import DataLoader
from dataset import NMTDataset, nmt_collate_fn, PAD_IDX, SOS_IDX, EOS_IDX
from transformers import PreTrainedTokenizerFast
import os

In [10]:
"dataset.py is just a copy of dataset.ipynb" 

'dataset.py is just a copy of dataset.ipynb'

In [12]:
def verify():
    '''Ensure the trained BPE tokenizer has been created and saved before loading it'''
    if not os.path.exists("tokenizer_target"):
        print("Error: Train tokenizers first by running 'python tokenizer.py'")
        return
    '''loads the trained BPE tokenize to get the tgt_inputs token ids'''
    tgt_tokenizer = PreTrainedTokenizerFast.from_pretrained("tokenizer_target")

    '''Path to the Hindi validation dataset'''
    CSV_PATH = "./nmtdataset/Hindi_dev.csv"
    if not os.path.exists(CSV_PATH):
        print(f"Error: Dataset not found at {CSV_PATH}")
        return
        
    dummy_word2idx = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}
    '''If a saved GloVe vocabulary exists, load it'''
    if os.path.exists("glove_vocab.pt"):
        '''Load the saved GloVe vocabulary checkpoint'''
        checkpoint = torch.load("glove_vocab.pt")
        '''Replace the dummy vocabulary with the actual GloVe word-to-index mapping'''
        dummy_word2idx = checkpoint.get("word2idx", dummy_word2idx)
        print("Using actual GloVe vocab.")

    dataset = NMTDataset(CSV_PATH, tgt_tokenizer, glove_word2idx=dummy_word2idx)
    '''Create a DataLoader that loads and batches dataset samples for training'''
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=nmt_collate_fn)

    print(f"Dataset Size: {len(dataset)}")
    '''Fetch the first padded batch of samples from the DataLoader
       "src_ids": src_ids,
       "src_attention_mask": src_mask,
       "tgt_inputs": tgt_inputs, 
       "tgt_labels": tgt_labels,
       "tgt_attention_mask": tgt_mask,
    '''
    batch = next(iter(dataloader))
    
    src_ids = batch["src_ids"]
    tgt_inputs = batch["tgt_inputs"]
    tgt_labels = batch["tgt_labels"]
    
    print("\n--- Batch Verification ---")
    print(f"src_ids.shape:     {src_ids.shape}")
    print(f"tgt_inputs.shape:  {tgt_inputs.shape}")
    print(f"tgt_labels.shape:  {tgt_labels.shape}")
    
    print("\nVerifying Teacher Forcing label shift...")
    '''chooses the first sample in the batch.'''
    sample_idx = 0
    print(f"Sample {sample_idx} first 5 input ids:  {tgt_inputs[sample_idx, :5].tolist()}")
    print(f"Sample {sample_idx} first 5 label ids:  {tgt_labels[sample_idx, :5].tolist()}")
    '''Notice everything in tgt_labels is shifted left by one compared to tgt_inputs ?? ....hmm that's teach force'''

    '''Checks that the very first decoder input token is <SOS> ? -> should print true'''
    print(f"Input starts with SOS_IDX ({SOS_IDX}): {tgt_inputs[sample_idx, 0] == SOS_IDX}")
    
    eos_in_label = (tgt_labels[sample_idx] == EOS_IDX).nonzero()
    if len(eos_in_label) > 0:
        print(f"Found EOS in label at index: {eos_in_label[0].item()}")
    else:
        print("Warning: EOS not found in this sample batch (maybe too short or max_tgt_len hit).")
        
    pad_count = (tgt_labels == PAD_IDX).sum().item()
    print(f"Total PAD tokens in batch: {pad_count}")
    
    print("\n--- PHASE 1 VERIFICATION COMPLETE ---")

In [13]:
if __name__ == "__main__":
    verify()

Using actual GloVe vocab.
Dataset Size: 3242

--- Batch Verification ---
src_ids.shape:     torch.Size([4, 9])
tgt_inputs.shape:  torch.Size([4, 17])
tgt_labels.shape:  torch.Size([4, 17])

Verifying Teacher Forcing label shift...
Sample 0 first 5 input ids:  [2, 390, 1220, 1327, 320]
Sample 0 first 5 label ids:  [390, 1220, 1327, 320, 291]
Input starts with SOS_IDX (2): True
Found EOS in label at index: 8
Total PAD tokens in batch: 18

--- PHASE 1 VERIFICATION COMPLETE ---
